# Milestone A — Checkpoint 1
## Kontrak Input dan Provenance Tahap 2

Notebook ini menjalankan `Riset/scripts/checkpoint_01_prepare_input.py`, lalu memeriksa artefak dokumen dan laporan audit.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from collections import Counter
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "Riset").exists() else cwd.parent
if not (REPO_ROOT / "Riset").exists():
    raise FileNotFoundError("Root repositori tidak ditemukan.")

SCRIPT = REPO_ROOT / "Riset/scripts/checkpoint_01_prepare_input.py"
INPUT_FILE = REPO_ROOT / "Riset/crawler_alodokter/hasil_normalisasi_tahap_1.jsonl"
OUTPUT_FILE = REPO_ROOT / "Riset/Tahap_2/checkpoint_01/stage2_checkpoint_01_input.jsonl"
AUDIT_FILE = REPO_ROOT / "Riset/Tahap_2/checkpoint_01/stage2_checkpoint_01_audit.json"

print("Repository :", REPO_ROOT)
print("Script     :", SCRIPT)
print("Input      :", INPUT_FILE)
print("Output     :", OUTPUT_FILE)
print("Audit      :", AUDIT_FILE)

## 1. Validasi file yang diperlukan

In [ ]:
required = [SCRIPT, INPUT_FILE]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"File tidak ditemukan: {missing}")
print("File yang diperlukan tersedia.")

## 2. Jalankan skrip Checkpoint 1

In [ ]:
command = [
    sys.executable,
    str(SCRIPT),
    "--input-file", str(INPUT_FILE),
    "--output-file", str(OUTPUT_FILE),
    "--audit-file", str(AUDIT_FILE),
    "--expected-records", "150",
]
result = subprocess.run(
    command,
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
    check=False,
)
print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(f"Checkpoint gagal dengan exit code {result.returncode}")

## 3. Baca artefak dan laporan audit

In [ ]:
documents = [
    json.loads(line)
    for line in OUTPUT_FILE.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
audit = json.loads(AUDIT_FILE.read_text(encoding="utf-8"))

print("Status            :", audit["status"])
print("Input records     :", audit["record_counts"]["input_records"])
print("Output documents  :", audit["record_counts"]["output_documents"])
print("Question documents:", audit["record_counts"]["question_documents"])
print("Answer documents  :", audit["record_counts"]["answer_documents"])
print("Issues            :", len(audit["issues"]))

## 4. Periksa exit criteria

In [ ]:
for criterion, passed in audit["exit_criteria"].items():
    marker = "PASS" if passed else "FAIL"
    print(f"[{marker}] {criterion}")

assert audit["status"] == "pass"
assert all(audit["exit_criteria"].values())

## 5. Periksa distribusi dan pasangan provenance

In [ ]:
source_counts = Counter(document["source_field"] for document in documents)
speaker_counts = Counter(document["speaker"] for document in documents)
parent_counts = Counter(document["parent_record_id"] for document in documents)

print("Source :", dict(source_counts))
print("Speaker:", dict(speaker_counts))
print("Semua parent mempunyai dua dokumen:", all(count == 2 for count in parent_counts.values()))

assert source_counts == {"question": 150, "answer": 150}
assert speaker_counts == {"patient": 150, "doctor": 150}
assert all(count == 2 for count in parent_counts.values())
assert len({document["document_id"] for document in documents}) == 300

## 6. Tampilkan contoh question dan answer dari satu record

In [ ]:
sample_parent_id = documents[0]["parent_record_id"]
sample_pair = [
    document for document in documents
    if document["parent_record_id"] == sample_parent_id
]

for document in sample_pair:
    print("=" * 80)
    print("document_id :", document["document_id"])
    print("source_field:", document["source_field"])
    print("speaker     :", document["speaker"])
    print("sentences   :", document["sentence_count"])
    print("text        :", document["normalized_text"][:600])

## 7. Pemeriksaan akhir

In [ ]:
assert len(documents) == 300
assert all(document["raw_text"].strip() for document in documents)
assert all(document["normalized_text"].strip() for document in documents)
assert all(document["sentences"] for document in documents)
assert all(document["provenance"]["raw_text_sha256"] for document in documents)
assert all(document["provenance"]["normalized_text_sha256"] for document in documents)

print("Checkpoint 1 berhasil dan seluruh exit criteria terpenuhi.")
print("Artefak dokumen:", OUTPUT_FILE)
print("Laporan audit  :", AUDIT_FILE)